In [1]:
import os
import json
import pandas as pd

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
KEY=os.getenv("GOOGLE_API_KEY")

In [ ]:
llm = ChatGoogleGenerativeAI(google_api_key=KEY, model="gemini-2.5-flash", temperature=0.3, max_output_tokens=2048)

In [6]:
llm

ChatGoogleGenerativeAI(output_version=None, profile={'name': 'Gemini 2.5 Flash Lite', 'release_date': '2025-06-17', 'last_updated': '2025-06-17', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), location=None, model='gemini-2.5-flash-lite', temperature=0.3, max_output_tokens=2048, client=<google.genai.client.Client object at 0x000001CBD45BDBE0>, default_metadata=(), model_kwargs={})

In [7]:
from langchain_core.prompts import PromptTemplate

In [8]:
RESPONSE_JSON = {
    "1": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "2": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "3": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
}

In [9]:
TEMPLATE = """
Text:{text}
You are an expert MCQ maker. Given the above text, it is your job to \
create a quiz  of {number} multiple choice questions for {subject} students in {tone} tone. 
Make sure the questions are not repeated and check all the questions to be conforming the text as well.
Make sure to format your response like  RESPONSE_JSON below  and use it as a guide. \
Ensure to make {number} MCQs
### RESPONSE_JSON
{response_json}

"""

In [10]:
quiz_generation_prompt=PromptTemplate(
    input_variables=["text", "number", "subject", "tone", "response_json"],
    template=TEMPLATE
)

In [11]:
quiz_chain = quiz_generation_prompt | llm

In [12]:
TEMPLATE2="""
You are an expert {subject} teacher.

Here is a quiz:
{quiz}

Your task:
- Review each question
- Fix any incorrect questions or answers
- Improve clarity
- Ensure correctness

Return ONLY the corrected MCQs in the same format.
"""

In [13]:
quiz_evaluation_prompt=PromptTemplate(input_variables=["subject", "quiz"], template=TEMPLATE2)

In [14]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import JsonOutputParser

In [15]:
generate_evaluate_chain=(
    {
        "quiz": quiz_chain,                     # output of first chain
        "subject": RunnablePassthrough()   # pass subject forward
    }
    | quiz_evaluation_prompt
    | llm
    | {
        "raw": RunnablePassthrough(),   # keeps AIMessage
        "parsed": JsonOutputParser()    # parsed JSON
    }
)

In [16]:
file_path=r"..\data.txt"

In [17]:
with open(file_path, 'r') as file:
    TEXT = file.read()

In [18]:
TEXT

'The term machine learning was coined in 1959 by Arthur Samuel, an IBM employee and pioneer in the field of computer gaming and \nartificial intelligence. The synonym self-teaching computers was also used during this time period.\n\nThe earliest machine learning program was introduced in the 1950s when Arthur Samuel invented a computer program that calculated the winning \nchance in checkers for each side, but the history of machine learning roots back to decades of human desire and effort to study human \ncognitive processes. In 1949, Canadian psychologist Donald Hebb published the book The Organization of Behavior, in which he introduced a \ntheoretical neural structure formed by certain interactions among nerve cells. Hebb\'s model of neurons interacting with one another set a \ngroundwork for how AIs and machine learning algorithms work under nodes, or artificial neurons used by computers to communicate data. Other \nresearchers who have studied human cognitive systems contributed 

In [19]:
json.dumps(RESPONSE_JSON)

'{"1": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "2": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "3": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}}'

In [20]:
NUMBER=5 
SUBJECT="machine learning"
TONE="simple"

In [21]:
from langchain_core.globals import set_verbose
set_verbose(True)

In [22]:
response = generate_evaluate_chain.invoke({
    "text": TEXT,
    "number": NUMBER,
    "subject":SUBJECT,
    "tone": TONE,
    "response_json": json.dumps(RESPONSE_JSON)
})

In [23]:
response

{'raw': AIMessage(content='```json\n{\n  "1": {\n    "mcq": "Who is credited with coining the term \'machine learning\' in 1959?",\n    "options": {\n      "a": "Donald Hebb",\n      "b": "Arthur Samuel",\n      "c": "Raytheon Company",\n      "d": "Tom M. Mitchell"\n    },\n    "correct": "b"\n  },\n  "2": {\n    "mcq": "What was the name of the experimental \'learning machine\' developed by Raytheon Company in the early 1960s that could analyze sonar signals, electrocardiograms, and speech patterns?",\n    "options": {\n      "a": "Cybertron",\n      "b": "AlphaGo",\n      "c": "GANs",\n      "d": "The Organization of Behavior"\n    },\n    "correct": "a"\n  },\n  "3": {\n    "mcq": "According to Tom M. Mitchell\'s formal definition, a computer program is said to learn from experience E with respect to some class of tasks T and performance measure P if its performance at tasks in T, as measured by P, improves with what?",\n    "options": {\n      "a": "Cognitive terms",\n      "b": "

In [24]:
type(response["parsed"])

dict

In [25]:
response["parsed"]

{'1': {'mcq': "Who is credited with coining the term 'machine learning' in 1959?",
  'options': {'a': 'Donald Hebb',
   'b': 'Arthur Samuel',
   'c': 'Raytheon Company',
   'd': 'Tom M. Mitchell'},
  'correct': 'b'},
 '2': {'mcq': "What was the name of the experimental 'learning machine' developed by Raytheon Company in the early 1960s that could analyze sonar signals, electrocardiograms, and speech patterns?",
  'options': {'a': 'Cybertron',
   'b': 'AlphaGo',
   'c': 'GANs',
   'd': 'The Organization of Behavior'},
  'correct': 'a'},
 '3': {'mcq': "According to Tom M. Mitchell's formal definition, a computer program is said to learn from experience E with respect to some class of tasks T and performance measure P if its performance at tasks in T, as measured by P, improves with what?",
  'options': {'a': 'Cognitive terms',
   'b': 'Mathematical models',
   'c': 'Experience',
   'd': 'Human imitation'},
  'correct': 'c'},
 '4': {'mcq': 'Which of the following is NOT one of the three m

In [26]:
response["raw"].usage_metadata

{'input_tokens': 1628,
 'output_tokens': 526,
 'total_tokens': 2154,
 'input_token_details': {'cache_read': 0}}

In [27]:
usage = response["raw"].usage_metadata

print("Prompt Tokens:", usage.get("input_tokens"))
print("Completion Tokens:", usage.get("output_tokens"))
print("Total Tokens:", usage.get("total_tokens"))

Prompt Tokens: 1628
Completion Tokens: 526
Total Tokens: 2154


In [28]:
print(type(response))

<class 'dict'>


In [29]:
quiz=response["parsed"]

In [30]:
for key, value in quiz.items():
    print(f"{key}, {value.get("mcq")}")
    print(value.get("options").values())
    print(value.get("options").get(value.get("correct")))

1, Who is credited with coining the term 'machine learning' in 1959?
dict_values(['Donald Hebb', 'Arthur Samuel', 'Raytheon Company', 'Tom M. Mitchell'])
Arthur Samuel
2, What was the name of the experimental 'learning machine' developed by Raytheon Company in the early 1960s that could analyze sonar signals, electrocardiograms, and speech patterns?
dict_values(['Cybertron', 'AlphaGo', 'GANs', 'The Organization of Behavior'])
Cybertron
3, According to Tom M. Mitchell's formal definition, a computer program is said to learn from experience E with respect to some class of tasks T and performance measure P if its performance at tasks in T, as measured by P, improves with what?
dict_values(['Cognitive terms', 'Mathematical models', 'Experience', 'Human imitation'])
Experience
4, Which of the following is NOT one of the three main types of modern-day Machine Learning algorithms mentioned in the text?
dict_values(['Supervised Learning Algorithms', 'Generative Adversarial Networks', 'Unsuperv

In [35]:
quiz_table_data = []
for key, value in quiz.items():
    mcq = value["mcq"]
    options = " | ".join(
        [
            f"{option}: {option_value}"
            for option, option_value in value["options"].items()
            ]
        )
    correct = value["correct"]
    quiz_table_data.append({"MCQ": mcq, "Choices": options, "Correct": correct})

In [36]:
quiz_table_data

[{'MCQ': 'Who first coined the term "machine learning" and in what year?',
  'Choices': 'a: Donald Hebb in 1949 | b: Arthur Samuel in 1959 | c: Tom M. Mitchell in 1981 | d: Ian Goodfellow in 2014',
  'Correct': 'b'},
 {'MCQ': 'Which of these is NOT listed as an objective of current Unsupervised Learning Algorithms?',
  'Choices': 'a: Clustering | b: Classification | c: Dimensionality reduction | d: Association rule',
  'Correct': 'b'},
 {'MCQ': 'What was the primary function of the "Cybertron" learning machine developed by Raytheon Company in the early 1960s?',
  'Choices': 'a: To play checkers against human opponents | b: To recognize 40 characters from a computer terminal | c: To analyze sonar signals, electrocardiograms, and speech patterns | d: To introduce generative adversarial networks',
  'Correct': 'c'},
 {'MCQ': 'In Tom M. Mitchell\'s widely quoted definition of machine learning, what improves with "experience E"?',
  'Choices': 'a: The number of tasks (T) it can perform | b:

In [37]:
df=pd.DataFrame(quiz_table_data)

In [38]:
df

,MCQ,Choices,Correct
0,"Who first coined the term ""machine learning"" a...",a: Donald Hebb in 1949 | b: Arthur Samuel in 1...,b
1,Which of these is NOT listed as an objective o...,a: Clustering | b: Classification | c: Dimensi...,b
2,"What was the primary function of the ""Cybertro...",a: To play checkers against human opponents | ...,c
3,In Tom M. Mitchell's widely quoted definition ...,a: The number of tasks (T) it can perform | b:...,b
4,"Who published ""The Organization of Behavior"" i...",a: Walter Pitts | b: Warren McCulloch | c: Don...,c


In [39]:
df.to_csv("Machine_Learning_quiz.csv",index=False)

In [1]:
from datetime import datetime
LOG_FILE=f"{datetime.now().strftime('%m_%d_%Y_%H_%M_%S')}.log"

In [2]:
LOG_FILE

'04_26_2026_21_43_06.log'